# Bayesian Hyperparameter Space Optimization and Automated Model Auditing Pipeline

In [1]:
%load_ext autoreload
%autoreload 2

import os
import mlflow
import optuna
from optuna.integration.mlflow import MLflowCallback
from optuna.pruners import MedianPruner
import numpy as np
import pandas as pd

from src import feature_engineering as fe
from src import optuna_optimization as utils
from src import mlflow_utils as mlf_utils

/Users/hector.vargas/repos/ml_hands_on_project/env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Path Configuration & MLflow Backend Binding

In [2]:
RANDOM_STATE = 42
N_SPLITS = 5
EXPERIMENT_NAME = "customer-churn-optuna"

from pathlib import Path

def _find_project_root():
    """Find the project root by looking for pyproject.toml."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    raise FileNotFoundError("Could not find project root (pyproject.toml)")

ROOT_DIR = str(_find_project_root())
DB_PATH = os.path.join(ROOT_DIR, "mlflow.db")
ARTIFACTS_DIR = os.path.join(ROOT_DIR, "mlartifacts")

# Set the tracking URI immediately to lock it to SQLite
mlflow.set_tracking_uri(f"sqlite:///{DB_PATH}")

# Explicitly create the experiment with your designated mlartifacts folder path
experiment_id = mlf_utils.init_mlflow_experiment(
    EXPERIMENT_NAME,
    DB_PATH,
    ARTIFACTS_DIR,
)

# Active the experiment scope
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='file:///Users/hector.vargas/repos/ml_hands_on_project/mlartifacts', creation_time=1781781840960, experiment_id='1', last_update_time=1781781840960, lifecycle_stage='active', name='customer-churn-optuna', tags={}, trace_location=None, workspace='default'>

## 2. Custom Source Code Ingestion

In [3]:
X_train = pd.read_csv("../data/processed/raw_features/X_train.csv")
X_test = pd.read_csv("../data/processed/raw_features/X_test.csv")
y_train = pd.read_csv("../data/processed/target/y_train.csv").squeeze()
y_test = pd.read_csv("../data/processed/target/y_test.csv").squeeze()

## 2. Orchestrate and Initialize Search Parameter Studies

In [4]:
selected_features = [
    # Binary
    "is_silver",
    "is_germany",
    "is_spain",
    "Num_Of_Products_1",
    "Num_Of_Products_2",
    "Num_Of_Products_3",
    "Num_Of_Products_4",

    # Continuous
    "Age_x_IsActive",
    "Balance_per_Product",
    "CreditScore_per_Age",
    "Inactive_x_Balance",
    "CreditScore_x_Age",
    "Products_per_Tenure",
]
# Schema Baseline Columns Definitions
nomod_columns = []
dummyfy_columns = ['Gender']
norm_std_columns = ['Point Earned', 'Satisfaction Score', 'EstimatedSalary']

# Initialize the Feature Engineer class with the desired subset strings
feature_engineer_object = fe.DynamicFeatureEngineer(selected_features=selected_features)
binary_features = feature_engineer_object._get_all_binary_features()
continuous_features = feature_engineer_object._get_all_continuous_features()

# FIX: Set current_layout explicitly before instantiating your classes
current_layout = {
    "passthrough": nomod_columns + binary_features,
    "standard_scale": norm_std_columns + continuous_features,
    "one_hot_encode": dummyfy_columns
}

EXPERIMENT_REGISTRY = {
    "experiment_1": current_layout
}

# Initialize Integrated MLflow Tracking Integration Callbacks
mlflow_callback = MLflowCallback(
    tracking_uri=mlflow.get_tracking_uri(),
    metric_name="pr_auc",
    create_experiment=False,
    mlflow_kwargs={
        "nested": True,
        "experiment_id": experiment_id,
    },
)

study = optuna.create_study(
    study_name="customer-churn-rf-search-v1",
    direction="maximize",
    pruner=MedianPruner(n_startup_trials=5, n_warmup_steps=0),
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
)

# FIXED: Removed the invalid FULL_REGISTRY argument variable completely
# Instantiate the objective function with both training and testing datasets
objective_function = utils.ObjectiveCV(
    X=X_train,
    y=y_train,
    current_layout=current_layout,
    n_splits=N_SPLITS,
    random_state=RANDOM_STATE,
)

/var/folders/bv/50x24wc545x5mclk_t88ryrc0000gn/T/ipykernel_18808/1381256263.py:41: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
[I 2026-06-18 05:28:51,263] A new study created in memory with name: customer-churn-rf-search-v1


## 3. Run Optimization Search Workspace Execution Window

In [5]:
# Execute the parent run context block
with mlflow.start_run(
    run_name="optuna_search_parent",
    experiment_id=experiment_id,
):    
    study.optimize(
        objective_function,
        n_trials=10,
        callbacks=[mlflow_callback]
    )

    # Log global baseline metrics directly to the parent run metadata shell
    mlflow.log_metric("best_auc", study.best_value)
    mlflow.log_params(study.best_params)

    # Reconstruct the pipeline architecture using the best parameters found
    best_pipeline = utils.build_pipeline(
        trial=study.best_trial, 
        current_layout=current_layout,
        random_state=RANDOM_STATE
    )

    # Run full evaluations and save serialization schema artifacts to the parent run
    utils.evaluate_and_log_best_model(
        best_pipeline=best_pipeline,
        X_train=X_train, 
        y_train=y_train,
        X_test=X_test, 
        y_test=y_test,
        current_layout=current_layout,
    )

[I 2026-06-18 05:28:54,296] Trial 0 finished with value: 0.6788379596098221 and parameters: {'scaler': 'minmax', 'encoder': 'drop_first', 'model': 'rf', 'rf_n_estimators': 293, 'rf_max_depth': 12, 'rf_min_samples_split': 25, 'rf_min_samples_leaf': 1}. Best is trial 0 with value: 0.6788379596098221.
[I 2026-06-18 05:28:55,794] Trial 1 finished with value: 0.6953347996684984 and parameters: {'scaler': 'std', 'encoder': 'no_drop', 'model': 'xgb', 'xgb_n_estimators': 1059, 'xgb_learning_rate': 0.029667626364545993, 'xgb_subsample': 0.9644741157888952, 'xgb_colsample_bytree': 0.47092407909780626, 'xgb_gamma': 2.0842892970704363, 'xgb_reg_alpha': 6.78905327169848e-07, 'xgb_reg_lambda': 1.9069966103000426e-07, 'xgb_scale_pos_weight': 1.992587980696507}. Best is trial 1 with value: 0.6953347996684984.
[I 2026-06-18 05:28:57,893] Trial 2 finished with value: 0.6800919420716183 and parameters: {'scaler': 'robust', 'encoder': 'no_drop', 'model': 'rf', 'rf_n_estimators': 295, 'rf_max_depth': 13, '

## 4. Display Session Optimization Diagnostics Results

In [6]:
print(f"\nTop Optimization Average Precision Score Target: {study.best_value:.4f}")
print("\nOptimal Parameter Combinations Selected:")
for parameter_key, parameter_value in study.best_params.items():
    print(f" * {parameter_key}: {parameter_value}")


Top Optimization Average Precision Score Target: 0.6961

Optimal Parameter Combinations Selected:
 * scaler: robust
 * encoder: no_drop
 * model: xgb
 * xgb_n_estimators: 1264
 * xgb_learning_rate: 0.02809743391955222
 * xgb_subsample: 0.940220884684944
 * xgb_colsample_bytree: 0.5723192142682251
 * xgb_gamma: 2.9137146876952342
 * xgb_reg_alpha: 4.416068895118589e-05
 * xgb_reg_lambda: 7.183758655813991e-06
 * xgb_scale_pos_weight: 1.6370223258670453


In [7]:
suggestions = utils.suggest_numeric_ranges(study)
display(suggestions)

,parameter,current_min,current_max,best_median,action,suggested_min,suggested_max
0,xgb_learning_rate,2.809743e-02,0.038387,0.028097,move_left,2.295246e-02,0.038387
1,xgb_reg_lambda,1.126976e-07,0.000070,0.000007,move_left,-3.505937e-05,0.000070
2,xgb_scale_pos_weight,1.637022e+00,1.992588,1.637022,move_left,1.459239e+00,1.992588
3,xgb_subsample,9.402209e-01,0.965502,0.940221,move_left,9.275802e-01,0.965502
4,xgb_colsample_bytree,4.709241e-01,0.583082,0.572319,move_right,4.709241e-01,0.639161
5,xgb_n_estimators,8.180000e+02,1264.000000,1264.000000,move_right,8.180000e+02,1487.000000
6,xgb_reg_alpha,3.962518e-08,0.000044,0.000044,move_right,3.962518e-08,0.000066
7,xgb_gamma,2.084289e+00,3.315133,2.913715,narrow,2.913715e+00,2.913715


In [8]:
import mlflow
import pandas as pd

experiment = mlflow.get_experiment_by_name(
    "customer-churn-optuna"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.average_precision DESC"]
)

runs.head()

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.pr_auc,metrics.test_roc_auc,metrics.test_recall,metrics.train_roc_auc,...,tags.mlflow.user,tags.direction,tags.xgb_gamma_distribution,tags.xgb_colsample_bytree_distribution,tags.xgb_reg_lambda_distribution,tags.state,tags.rf_min_samples_split_distribution,tags.rf_n_estimators_distribution,tags.rf_max_depth_distribution,tags.rf_min_samples_leaf_distribution
0,9fc75c2e8af348f5a2e8b06f88e76fe5,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 11:29:05.777000+00:00,2026-06-18 11:29:05.796000+00:00,0.690276,NaN,NaN,NaN,...,hector.vargas,MAXIMIZE,"FloatDistribution(high=3.5, log=False, low=1.5...","FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.0001, log=True, low=1...",PRUNED,None,None,None,None
1,a813e426d12a40ca8d4349055b0feec1,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 11:29:03.968000+00:00,2026-06-18 11:29:03.982000+00:00,0.677898,NaN,NaN,NaN,...,hector.vargas,MAXIMIZE,None,None,None,PRUNED,"IntDistribution(high=28, log=False, low=18, st...","IntDistribution(high=295, log=False, low=280, ...","IntDistribution(high=13, log=False, low=9, ste...","IntDistribution(high=3, log=False, low=1, step=1)"
2,df8a58936ea9477096bb64e1e4f7ea86,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 11:29:02.671000+00:00,2026-06-18 11:29:02.689000+00:00,0.695085,NaN,NaN,NaN,...,hector.vargas,MAXIMIZE,"FloatDistribution(high=3.5, log=False, low=1.5...","FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.0001, log=True, low=1...",COMPLETE,None,None,None,None
3,bb9bed36659b41df808e95b571827117,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 11:29:01.032000+00:00,2026-06-18 11:29:01.047000+00:00,0.693027,NaN,NaN,NaN,...,hector.vargas,MAXIMIZE,"FloatDistribution(high=3.5, log=False, low=1.5...","FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.0001, log=True, low=1...",COMPLETE,None,None,None,None
4,a60d2371aa5f4f6eb896196c10c2b55d,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 11:29:00.656000+00:00,2026-06-18 11:29:00.669000+00:00,0.696080,NaN,NaN,NaN,...,hector.vargas,MAXIMIZE,"FloatDistribution(high=3.5, log=False, low=1.5...","FloatDistribution(high=0.6, log=False, low=0.4...","FloatDistribution(high=0.0001, log=True, low=1...",COMPLETE,None,None,None,None


In [9]:
run_id = runs.iloc[0]["run_id"]

run = mlflow.get_run(run_id)

print(run.data.metrics)

{'pr_auc': 0.6902761328074174}


In [10]:
runs_df = mlflow.search_runs()
# Select rows where 'tags.mlflow.parentRunId' is missing (meaning it IS a parent/master run)
parent_summary_df = runs_df[runs_df["tags.mlflow.parentRunId"].isna() & (runs_df["status"] == "FINISHED")]

# Isolate evaluation metrics
metric_cols = [c for c in parent_summary_df.columns if c.startswith("metrics.")]
display(parent_summary_df[["start_time"] + metric_cols])

,start_time,metrics.pr_auc,metrics.test_roc_auc,metrics.test_recall,metrics.train_roc_auc,metrics.train_recall,metrics.test_pr_auc,metrics.train_precision,metrics.best_auc,metrics.train_pr_auc,metrics.test_accuracy,metrics.train_accuracy,metrics.test_f1,metrics.test_precision,metrics.train_f1
10,2026-06-18 11:28:51.412000+00:00,NaN,0.874386,0.583333,0.94805,0.68865,0.733325,0.800357,0.69608,0.849821,0.8625,0.901563,0.633822,0.693878,0.740313
21,2026-06-18 11:24:36.134000+00:00,NaN,0.874386,0.583333,0.94805,0.68865,0.733325,0.800357,0.69608,0.849821,0.8625,0.901563,0.633822,0.693878,0.740313


In [11]:
parent_summary_df

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.pr_auc,metrics.test_roc_auc,metrics.test_recall,metrics.train_roc_auc,...,tags.mlflow.user,tags.direction,tags.xgb_gamma_distribution,tags.xgb_colsample_bytree_distribution,tags.xgb_reg_lambda_distribution,tags.state,tags.rf_min_samples_split_distribution,tags.rf_n_estimators_distribution,tags.rf_max_depth_distribution,tags.rf_min_samples_leaf_distribution
10,5f2fd1dc86194a0f89a9555b199fa9cf,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 11:28:51.412000+00:00,2026-06-18 11:29:09.127000+00:00,NaN,0.874386,0.583333,0.94805,...,hector.vargas,None,None,None,None,None,None,None,None,None
21,912ffa2b5d69402e85318243adf7ce03,1,FINISHED,file:///Users/hector.vargas/repos/ml_hands_on_...,2026-06-18 11:24:36.134000+00:00,2026-06-18 11:24:53.337000+00:00,NaN,0.874386,0.583333,0.94805,...,hector.vargas,None,None,None,None,None,None,None,None,None


In [12]:
# from src import feature_engineering as fe
# fe.remove_recent_runs(EXPERIMENT_NAME, 100000)

In [13]:
rf_runs = (
    runs[runs["params.model"] == "rf"]
    .sort_values(
        "metrics.train_pr_auc",
        ascending=False
    )
)

rf_runs[
    [
        "run_id",
        'metrics.train_pr_auc',
        "metrics.pr_auc",
        "params.rf_max_depth",
        "params.rf_n_estimators",
        "params.rf_min_samples_split",
        "params.rf_min_samples_leaf",
    ]
]

,run_id,metrics.train_pr_auc,metrics.pr_auc,params.rf_max_depth,params.rf_n_estimators,params.rf_min_samples_split,params.rf_min_samples_leaf
1,a813e426d12a40ca8d4349055b0feec1,NaN,0.677898,13,293,20,3
5,714be0bdbc414bc58ad5e8a969ddc96d,NaN,0.676056,10,285,20,3
7,61febe205657441a82f233ee2a2debb1,NaN,0.680092,13,295,26,1
9,8aee0a3f983144d9aa8dd8385644fa89,NaN,0.678838,12,293,25,1
12,57047eb8d27c48b8900c4525fafcf490,NaN,0.677898,13,293,20,3
16,35a814b41b7e4653994de6531303540a,NaN,0.676056,10,285,20,3
18,5db0d4a6e0424f1798362d811c5826b0,NaN,0.680092,13,295,26,1
20,f1b199ea1da441c8bb7c679a5e49998d,NaN,0.678838,12,293,25,1


In [15]:
xgb_runs = (
    runs[runs["params.model"] == "xgb"]
    .sort_values(
        "metrics.train_pr_auc",
        ascending=False
    )
)

xgb_runs[
    [
        "run_id",
        "metrics.train_pr_auc",
        "metrics.test_pr_auc",
        "metrics.pr_auc",
        "params.xgb_n_estimators",
        "params.xgb_learning_rate",
        "params.xgb_gamma",
        "params.xgb_scale_pos_weight",
    ]
].head(20)

,run_id,metrics.train_pr_auc,metrics.test_pr_auc,metrics.pr_auc,params.xgb_n_estimators,params.xgb_learning_rate,params.xgb_gamma,params.xgb_scale_pos_weight
10,5f2fd1dc86194a0f89a9555b199fa9cf,0.849821,0.733325,NaN,1264,0.02809743391955222,2.9137146876952342,1.6370223258670453
21,912ffa2b5d69402e85318243adf7ce03,0.849821,0.733325,NaN,1264,0.02809743391955222,2.9137146876952342,1.6370223258670453
0,9fc75c2e8af348f5a2e8b06f88e76fe5,NaN,NaN,0.690276,1291,0.041463013288466695,2.334822006297558,1.768807585701814
2,df8a58936ea9477096bb64e1e4f7ea86,NaN,NaN,0.695085,818,0.03634110166546641,3.3151329478521863,1.9777755692715244
3,bb9bed36659b41df808e95b571827117,NaN,NaN,0.693027,995,0.03838737414135848,2.444429850323899,1.9803925243084488
4,a60d2371aa5f4f6eb896196c10c2b55d,NaN,NaN,0.696080,1264,0.02809743391955222,2.9137146876952342,1.6370223258670453
6,aac544acc3b84535bea25a5cbe9ccfed,NaN,NaN,0.690538,955,0.03690317493012618,2.5934205586865593,1.9875664116805574
8,3d41751058f049f1a8fb485da0600d84,NaN,NaN,0.695335,1059,0.029667626364545993,2.0842892970704363,1.992587980696507
11,42a5f9c25aca424d9663145b52f726fa,NaN,NaN,0.690276,1291,0.041463013288466695,2.334822006297558,1.768807585701814
13,cc598681158e4b5880b2d764177be4ba,NaN,NaN,0.695085,818,0.03634110166546641,3.3151329478521863,1.9777755692715244
